# Phase 3: Time-Series Feature Engineering & Leakage Prevention
**Project:** Retail Sales & Demand Forecasting (FUTURE_ML_01)  
**Objective:** Transform daily aggregated sales into an ML-ready feature matrix with target lags, shifted rolling statistics, calendar indicators, holiday proximity, and strict chronological train/validation/test partitions.

## 1. Imports & Setup
We import components from `src.features` and our existing data pipeline.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.features import (
    load_daily_sales,
    create_lag_features,
    create_rolling_features,
    create_calendar_features,
    create_holiday_features,
    build_feature_matrix,
    chronological_train_test_split,
    export_featured_data,
)

pd.set_option('display.max_columns', 35)
pd.set_option('display.width', 1000)
%matplotlib inline
sns.set_theme(style="whitegrid", palette="deep")

## 2. Ingestion of Cleaned Daily Sales Data
We load the 374-day continuous daily sales series produced in Phase 2.

In [ ]:
daily_df = load_daily_sales()
print(f"Loaded daily sales: {len(daily_df)} days from {daily_df.index.min().date()} to {daily_df.index.max().date()}")
daily_df[['net_sales', 'gross_sales', 'returns_amount', 'net_quantity', 'total_orders']].head()

## 3. Time-Series Lag Features
We generate historical lags of `net_sales`:
- `lag_1`: Sales from yesterday (immediate autoregression)
- `lag_7`: Sales from same day last week (capturing weekly seasonality)
- `lag_14`: Sales from two weeks ago
- `lag_28`: Sales from four weeks ago (monthly cycle)

In [ ]:
lags_df = create_lag_features(daily_df, target_col='net_sales', lags=[1, 7, 14, 28])
lags_df.head(10)

## 4. Rolling Statistics & Data Leakage Prevention
**CRITICAL DESIGN RULE:** To prevent lookahead leakage, rolling statistics (mean and std) are calculated on `shift(1)`.

For any date $t$, the rolling statistics represent:
$$\text{rolling\_mean}_{w}(t) = \frac{1}{w} \sum_{i=1}^{w} \text{target}(t - i)$$

The target observation at date $t$ is **never** included in the calculation of the features for date $t$.

In [ ]:
rolling_df = create_rolling_features(daily_df, target_col='net_sales', windows=[7, 14, 28])
rolling_df.head(10)

## 5. Calendar & Holiday Features
We extract temporal indicators:
- `day_of_week`, `day_of_month`, `month`, `quarter`, `year`
- `is_weekend`, `is_saturday` (0-sales operational closure)
- Cyclical sine/cosine encodings for periodic boundaries
- `days_until_holiday`, `days_since_holiday`, `is_holiday` based on UK bank holidays and retail events.

In [ ]:
cal_df = create_calendar_features(daily_df)
hol_df = create_holiday_features(daily_df)
cal_hol_combined = pd.concat([cal_df, hol_df], axis=1)
cal_hol_combined.head(10)

## 6. Full Feature Matrix Assembly & Warmup Trimming
Because the maximum historical window is 28 days, the first 28 calendar days have NaN values. We drop these initial 28 warmup rows, leaving 346 clean rows with **0 missing values**.

In [ ]:
feature_matrix = build_feature_matrix(daily_df, target_col='net_sales', drop_warmup=True)
export_path = export_featured_data(feature_matrix)
print(f"Feature matrix dimensions: {feature_matrix.shape[0]} rows, {feature_matrix.shape[1]} columns")
print(f"Total missing values: {feature_matrix.isnull().sum().sum()}")
feature_matrix.head()

## 7. Feature Correlation Analysis
We examine how strongly the engineered lag and rolling features correlate with target daily net sales.

In [ ]:
corrs = feature_matrix.corr()['net_sales'].sort_values(ascending=False)
print("Top 10 Positively Correlated Features with Net Sales:")
print(corrs.head(10))

plt.figure(figsize=(10, 6))
top_features = corrs.drop(['net_sales', 'gross_sales', 'net_quantity', 'total_orders']).head(8)
sns.barplot(x=top_features.values, y=top_features.index, palette='Blues_r', edgecolor='black')
plt.title("Top Engineered Features Correlated with Net Sales (£)", weight='bold', pad=12)
plt.xlabel("Pearson Correlation Coefficient (r)")
plt.tight_layout()
plt.show()

### Correlation Insights:
- `net_sales_lag_7` shows strong positive correlation ($r \approx 0.52$), confirming pronounced 7-day weekly purchasing habits.
- `net_sales_rolling_mean_7` and `rolling_mean_14` provide smooth trend tracking ($r \approx 0.55$), capturing seasonal momentum.
- `is_saturday` exhibits strong negative correlation ($r = -0.47$) because the business is closed on Saturdays.

## 8. Chronological Train / Validation / Test Splitting
We partition the 346 clean daily records strictly in time order:
- **Train (70%):** 242 days (Dec 29, 2010 to Aug 27, 2011)
- **Validation (15%):** 51 days (Aug 28, 2011 to Oct 17, 2011) - for hyperparameter tuning
- **Test (15%):** 53 days (Oct 18, 2011 to Dec 09, 2011) - out-of-sample holiday evaluation

In [ ]:
train_df, val_df, test_df = chronological_train_test_split(feature_matrix, train_ratio=0.70, val_ratio=0.15, test_ratio=0.15)

split_summary = pd.DataFrame({
    'Partition': ['Train (70%)', 'Validation (15%)', 'Test (15%)'],
    'Days Count': [len(train_df), len(val_df), len(test_df)],
    'Start Date': [train_df.index.min().date(), val_df.index.min().date(), test_df.index.min().date()],
    'End Date': [train_df.index.max().date(), val_df.index.max().date(), test_df.index.max().date()],
    'Mean Net Sales': [f"GBP {train_df['net_sales'].mean():,.2f}", f"GBP {val_df['net_sales'].mean():,.2f}", f"GBP {test_df['net_sales'].mean():,.2f}"],
    'Total Net Sales': [f"GBP {train_df['net_sales'].sum():,.2f}", f"GBP {val_df['net_sales'].sum():,.2f}", f"GBP {test_df['net_sales'].sum():,.2f}"]
})
display(split_summary)

### Partition Timeline Visualization

In [ ]:
plt.figure(figsize=(13, 5))
plt.plot(train_df.index, train_df['net_sales'], label='Train (70%)', color='#1976D2', linewidth=1.5)
plt.plot(val_df.index, val_df['net_sales'], label='Validation (15%)', color='#FB8C00', linewidth=1.5)
plt.plot(test_df.index, test_df['net_sales'], label='Test (15%)', color='#E53935', linewidth=1.5)

plt.title("Chronological Train / Validation / Test Partitions (No Lookahead Leakage)", weight='bold', pad=12)
plt.xlabel("Date")
plt.ylabel("Net Sales (GBP)")
plt.legend(loc="upper left", frameon=True)
plt.tight_layout()
plt.show()

## 9. Conclusion & Readiness for Phase 4 (Forecasting Models)
1. **31 Leak-Free Features:** Fully engineered historical lags, shifted rolling aggregates, cyclical calendar features, and holiday distances.
2. **Clean Matrices:** 346 observations with 0 NaNs saved in `data/processed/featured_daily_sales.csv`.
3. **Chronological Splitting Verified:** Clean train/val/test partitions ready for baseline heuristics (Naive, Moving Average) and machine learning models (Ridge, Random Forest, Gradient Boosting) in Phase 4.